# Data Engineering: Pipeline de Reportería Automatizada - Contact Center Cloud Services

Este Notebook describe la arquitectura y generación de la base de datos **BD_BPO_CONTACT**. El enfoque principal es la creación de un ecosistema de datos robusto que permita medir la productividad y efectividad en la venta de servicios de **Google Cloud (Storage)**, utilizando visualizaciones avanzadas en Power BI con soporte de HTML5.

## 1. Estructura de la Base de Datos

La base de datos se consolidará en un archivo **XLSX** con cuatro dimensiones principales, garantizando la integridad referencial mediante el uso de **PK (Primary Keys)** y **FK (Foreign Keys)**.

### Hoja 1: `Calls_contact` (Transaccional de Llamadas)
Esta es la tabla de hechos que registra la actividad diaria de la operación.

| Encabezado | Tipo de Dato | Descripción |
| :--- | :--- | :--- |
| **Call_ID** | `INT (PK)` | Identificador único autoincremental de la llamada. |
| **Date_Contact** | `DATETIME` | Fecha y hora de inicio de la interacción (2023 - Q1 2026). |
| **User_ID** | `VARCHAR (FK)` | ID del agente (coincide con `Users_register`). |
| **Session_Type** | `ENUM` | Tipo de sesión: Inbound, Outbound, Seguimiento. |
| **Duration_Seconds** | `INT` | Duración total del contacto en segundos. |
| **Call_Status** | `ENUM` | Tipificación: Efectiva (Venta), No Efectiva, No Contactado. |
| **Session_Comments** | `TEXT` | Observaciones detalladas del agente en español. |
| **Service_Tag** | `VARCHAR` | Producto específico (ej. Google One, Drive, GCP Storage). |

### Hoja 2: `Users_register` (Maestra de Agentes)
Dimensionamiento de la fuerza de trabajo (Límite: **600 usuarios activos**).

| Encabezado | Tipo de Dato | Descripción |
| :--- | :--- | :--- |
| **User_ID** | `VARCHAR (PK)` | Código único del colaborador. |
| **Full_Name** | `VARCHAR` | Nombre completo del agente (Español). |
| **Team_Leader** | `VARCHAR` | Supervisor a cargo del equipo. |
| **Hire_Date** | `DATE` | Fecha de ingreso a la operación. |
| **Agent_Status** | `BOOLEAN` | Estado del usuario (Activo/Inactivo). |

### Hoja 3: `QA_report` (Calidad y Auditoría)
Métricas de cumplimiento y monitoreo de procesos.

| Encabezado | Tipo de Dato | Descripción |
| :--- | :--- | :--- |
| **QA_ID** | `INT (PK)` | ID único de la auditoría de calidad. |
| **Call_ID** | `INT (FK)` | Referencia a la llamada auditada. |
| **QA_Score** | `DECIMAL` | Calificación de calidad (0.0 - 100.0). |
| **Compliance_Status** | `BOOLEAN` | Indica si cumple o no con el script legal de Google Cloud. |
| **Feedback_Notes** | `TEXT` | Comentarios del analista de calidad. |

### Hoja 4: `NPS_score_call` (Satisfacción del Cliente)
Resultados de la encuesta post-llamada para medir lealtad (NPS) y satisfacción (CSAT).

| Encabezado | Tipo de Dato | Descripción |
| :--- | :--- | :--- |
| **Survey_ID** | `INT (PK)` | ID único de la encuesta. |
| **Call_ID** | `INT (FK)` | Referencia a la llamada original. |
| **NPS_Score** | `INT` | Calificación de 0 a 10 (Promotores, Pasivos, Detractores). |
| **CSAT_Score** | `INT` | Calificación de satisfacción de 1 a 5. |
| **Customer_Feedback** | `TEXT` | Voz del cliente (VOC) sobre el servicio recibido. |

---

## 2. Objetivos de la Estrategia de Seguimiento

1.  **Automatización de Reportería:** Eliminar el procesamiento manual de datos mediante scripts que generen el consolidado masivo (proyectado a un volumen de carga de alto tráfico).
2.  **Productividad y Ventas:** Analizar la correlación entre la **Duración de la Llamada** y la **Efectividad**, enfocándose en el cierre de ventas de almacenamiento en la nube.
3.  **Visualización Moderna:** Preparar los datos para una interfaz en **Power BI** que utilice DAX y objetos visuales personalizados (HTML/CSS) para reflejar la identidad de marca de servicios Cloud.

> **Nota Técnica:** Para cumplir con el volumen de registros solicitado hasta 2026, el script de generación aleatoria utilizará librerías como `pandas`, `numpy` y `faker` para asegurar coherencia en los comentarios y fechas.



# Preparacion de las librerias


## 1. Pandas 

Es, básicamente, excel en python, Si tenemos datos en filas y columnas , podemos usar **Pandas** para:
**Objetivo:** Manipular y analizar estructuras de datos de forma tabular.
**Utilidad:** Cargar archivos en formatos como (CVS, Excel,  SQL), filtrar información. limpiar datos faltantes y agrupar resultados de manera dinamica.

## 2. NumPy 

Es el motor matemático que corre detrás de casi todo en la ciencia de datos. Su nombre viene de **Numerical Python**

**Objetivo:** Realizar cálculos numéricos a alta velocidad.
**Utilidad:** Maneja grandes listas de números y aplica operaciones matemáticas complejas de forma mucho más rápida. 

## 3. Random 

Herramienta que usaremos para dar aleatoriedad a la infomación del archivo para el analisis. 

**Utilidad:** Simula lanzamientos de datos de manera aleatoria. Lo usaremos para no trabajar con datos reales aunque la estructura será una trabajada en una campaña de contact Center.

## 4. Datetime

Libreria para dar formato de fecha por medio de un calendario importado.

**Objetivo:** Crea de manera especifica campos de fecha en (Año, mes. dia), asi mismo permite hacer calculos dentro de nuestra informaicón.


In [ ]:
##pip install xlsxwriter
##pip install pandas
##pip install openpyxl
## python.exe -m pip install --upgrade pip

In [31]:
import pandas as pd
import numpy as np
import random
from datetime import datetime

# --- 1. CONFIGURACIÓN DE PARÁMETROS ---
NUM_USERS = 600
NUM_CALLS = 100000 
START_DATE = datetime(2023, 1, 1)
END_DATE = datetime(2026, 3, 31)

# --- 2. FUNCIÓN DE FECHAS ---
def random_dates(start, end, n):
    start_u = pd.to_datetime(start).value // 10**9
    end_u = pd.to_datetime(end).value // 10**9
    return pd.to_datetime(np.random.randint(start_u, end_u, n), unit='s')

# --- 3. GENERACIÓN DE DATOS MAESTROS ---
user_ids = [f"AGT_{str(i).zfill(3)}" for i in range(1, NUM_USERS + 1)]
names = ["Juan Perez", "Maria Garcia", "Carlos Lopez", "Ana Martinez", "Luis Rodriguez", "Elena Gomez"]
tls = [f"Sup_Cloud_{str(i).zfill(2)}" for i in range(1, 11)]

# --- 4. HOJA 2: Users_register ---
df_users = pd.DataFrame({
    'User_ID': user_ids,
    'Full_Name': [f"{random.choice(names)} {i}" for i in range(1, NUM_USERS + 1)],
    'Team_Leader': [random.choice(tls) for _ in range(NUM_USERS)],
    'Hire_Date': random_dates(pd.to_datetime('2022-01-01'), pd.to_datetime('2023-12-31'), NUM_USERS).date,
    'Agent_Status': [True] * int(NUM_USERS * 0.9) + [False] * (NUM_USERS - int(NUM_USERS * 0.9))
})

# --- 5. HOJA 1: Calls_contact ---
session_types = ['Inbound', 'Outbound', 'Follow-up']
statuses = ['Efectiva (Venta)', 'No Efectiva', 'No Contactado']
services = ['Google One 100GB', 'Google One 2TB', 'GCP Storage Nearline', 'GCP Coldline']
comments = ["Interesado en Storage", "Venta cerrada con éxito", "Llamada caída", "Consulta técnica cloud", "No interesado por precio"]

df_calls = pd.DataFrame({
    'Call_ID': np.arange(1, NUM_CALLS + 1),
    'Date_Contact': random_dates(START_DATE, END_DATE, NUM_CALLS),
    'User_ID': [random.choice(user_ids) for _ in range(NUM_CALLS)],
    'Session_Type': [random.choice(session_types) for _ in range(NUM_CALLS)],
    'Duration_Seconds': np.random.randint(30, 900, NUM_CALLS),
    'Call_Status': [random.choice(statuses) for _ in range(NUM_CALLS)],
    'Session_Comments': [random.choice(comments) for _ in range(NUM_CALLS)],
    'Service_Tag': [random.choice(services) for _ in range(NUM_CALLS)]
})

# --- 6. HOJA 3: QA_report (Corregida) ---
qa_count = int(NUM_CALLS * 0.85)
qa_call_ids = random.sample(list(df_calls['Call_ID']), qa_count)
feedback_options = ["Cumple script legal", "Mejorar manejo de objeciones", "Excelente cierre"]

df_qa = pd.DataFrame({
    'QA_ID': np.arange(1, qa_count + 1),
    'Call_ID': qa_call_ids,
    'QA_Score': np.random.uniform(10, 100, qa_count).round(2),
    'Compliance_Status': [random.choice([True, False]) for _ in range(qa_count)],
    'Feedback_Notes': [random.choice(feedback_options) for _ in range(qa_count)]
})

# --- 7. HOJA 4: NPS_score_call (Corregida) ---
nps_count = int(NUM_CALLS * 0.30)
nps_call_ids = random.sample(list(df_calls['Call_ID']), nps_count)
nps_feedback_options = ["Buen servicio", "Muy técnico", "Satisfecho con el plan", "Atención rápida"]

df_nps = pd.DataFrame({
    'Survey_ID': np.arange(1, nps_count + 1),
    'Call_ID': nps_call_ids,
    'NPS_Score': np.random.randint(0, 11, nps_count),
    'CSAT_Score': np.random.randint(1, 6, nps_count),
    'Customer_Feedback': [random.choice(nps_feedback_options) for _ in range(nps_count)]
})

# --- 8. EXPORTACIÓN A EXCEL ---
file_name = 'BD_BPO_CONTACT.xlsx'
with pd.ExcelWriter(file_name, engine='xlsxwriter') as writer:
    df_calls.to_excel(writer, sheet_name='Calls_contact', index=False)
    df_users.to_excel(writer, sheet_name='Users_register', index=False)
    df_qa.to_excel(writer, sheet_name='QA_report', index=False)
    df_nps.to_excel(writer, sheet_name='NPS_score_call', index=False)

print(f"¡Listo! Archivo '{file_name}' generado sin errores de longitud.")

¡Listo! Archivo 'BD_BPO_CONTACT.xlsx' generado sin errores de longitud.
